<a href="https://colab.research.google.com/github/giolessa/100-Days-of-Data/blob/main/day_06_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## EXPLANATION

1. **Inspect before cleaning.** The four diagnostic commands are `df.info()`, `df.dtypes`, `df.isna().sum()` (nulls per column), and `df.duplicated().sum()` (duplicate rows).
2. **Nulls in Pandas are `NaN` (or `NaT` for dates).** `isna()` and `notna()` play the role of SQL's `IS NULL` and `IS NOT NULL`, returning `True`/`False` Boolean masks that you can use for filtering.
   - **Two ways to handle nulls:** `dropna()` removes rows. With `subset=["col"]`, it checks only that specific column, and with `how="all"`, it only drops a row if all its values are null.
   - `fillna(value)` fills them in. It can be a fixed value, the mean (`df["col"].mean()`), the median, or a dictionary with a specific value per column: `df.fillna({"city": "not provided"})`. It's the equivalent of SQL's `COALESCE`.
3. **Imputation is not neutral.** Filling with `0`, the mean, or the median alters the analysis results. The median is safer than the mean when outliers are present. Decide on a column-by-column basis and document your reasoning.
4. **Duplicates:** `duplicated()` flags repeated rows after the first occurrence as `True`. `drop_duplicates()` removes those rows. With `subset=["order"]`, the comparison is based solely on that column, and with `keep="last"`, you keep the last occurrence.
5. **Type casting (conversion):** `astype(float)` converts types, but throws an error if there is non-numeric text.
   - `pd.to_numeric(col, errors="coerce")` converts values and replaces anything that fails with `NaN`.
   - `pd.to_datetime(col, format="%d/%m/%Y", errors="coerce")` converts dates, turning invalid ones into `NaT`. The `format` parameter avoids ambiguity between day/month and month/day.
   - `int64` does not support `NaN`, so an integer column with nulls becomes `float64`, unless you use the nullable `"Int64"` dtype (with a capital `I`).
6. **Text preview (the full topic is covered in Module 7):** `.str.strip()` removes leading and trailing whitespace, while `.str.lower()` and `.str.title()` standardize letter casing. Without this, `"Ana"`, `"ana "`, and `"ANA"` count as three different customers.
7. **Recommended workflow order:** First standardize text and cast data types, then remove duplicates, and finally handle missing values. Standardizing can uncover hidden duplicates, and type casting can introduce new nulls.
8. **Immutability by default:** These methods return a new DataFrame and do not modify the original one in place. Therefore, you need to reassign it: `df = df.drop_duplicates()`. Save a backup with `df_original = df.copy()` before starting.

## EXAMPLE

In [12]:
import pandas as pd
import numpy as np

students = pd.DataFrame({
    "name": ["Lia", "davi", "Lia", None, "RITA"],
    "grade": ["8.5", "4.0", "8.5", "no grade", "6.5"],
    "submission_date": ["01/11/2026", "02/11/2026", "01/11/2026", "03/11/2026", "30/02/2026"],
})

students_original = students.copy()
students.isna().sum()         # counts null values per column / soma os valores nulos por coluna
students.duplicated().sum()   # counts duplicate rows / soma as linhas repetidas

np.int64(1)

In [13]:
#  Trims whitespace and capitalizes each word / remove espaços nas pontas e deixa a inicial maiúscula

students["name"] = students["name"].str.strip().str.title()
print(students)


   name     grade submission_date
0   Lia       8.5      01/11/2026
1  Davi       4.0      02/11/2026
2   Lia       8.5      01/11/2026
3  None  no grade      03/11/2026
4  Rita       6.5      30/02/2026


In [14]:
# converts to float, turning invalid strings into NaN / converte para número, virando NaN o que não for válido

students["grade"] = pd.to_numeric(students["grade"], errors="coerce")
print(students)

   name  grade submission_date
0   Lia    8.5      01/11/2026
1  Davi    4.0      02/11/2026
2   Lia    8.5      01/11/2026
3  None    NaN      03/11/2026
4  Rita    6.5      30/02/2026


In [15]:
# parses DD/MM/YYYY dates, turning invalid dates like 30/02/2026 into NaT / converte datas DD/MM/YYYY, virando NaT as inválidas

students["submission_date"] = pd.to_datetime(students["submission_date"], format="%d/%m/%Y", errors="coerce")
print(students)

   name  grade submission_date
0   Lia    8.5      2026-11-01
1  Davi    4.0      2026-11-02
2   Lia    8.5      2026-11-01
3  None    NaN      2026-11-03
4  Rita    6.5             NaT


In [16]:
# drops exact duplicate rows, keeping the first occurrence by default / remove linhas duplicadas exatas, mantendo a primeira por padrão

students = students.drop_duplicates()
print(students)

   name  grade submission_date
0   Lia    8.5      2026-11-01
1  Davi    4.0      2026-11-02
3  None    NaN      2026-11-03
4  Rita    6.5             NaT


In [19]:
# fills missing values in name with "Unknown" ("desconhecido") / preenche valores nulos em name com "desconhecido" ("Unknown")

# students["name"] = students["name"].fillna("Unknown")

# or

students.loc[:, "name"] = students["name"].fillna("Unknown")

# assigns safely via .loc / atribui com segurança usando .loc

print(students)

      name  grade submission_date
0      Lia    8.5      2026-11-01
1     Davi    4.0      2026-11-02
3  Unknown    NaN      2026-11-03
4     Rita    6.5             NaT


In [22]:
# fills missing values in name with median / preenche valores nulos em name com a mediana

# studens["grade"] = students["grade"].fillna(students["grade"].median())

# or

students.loc[:, "grade"] = students["grade"].fillna(students["grade"].median())

# fills missing grades with the median using .loc / preenche notas ausentes com a mediana usando .loc

print(students)

      name  grade submission_date
0      Lia    8.5      2026-11-01
1     Davi    4.0      2026-11-02
3  Unknown    6.5      2026-11-03
4     Rita    6.5             NaT


## EXERCISE

In [63]:
import pandas as pd
import numpy as np

data = {
    "order_id": [201, 202, 203, 203, 204, 205, 206, 207],
    "customer": ["Ana", " bruno", "Carla", "Carla", "DIEGO", None, "Elisa", "ana "],
    "value": ["49.9", "249.9", "n/a", "n/a", "119.9", "35.0", "abc", "99.8"],
    "date": ["10/11/2026", "11/11/2026", "12/11/2026", "12/11/2026",
             "13/11/2026", "31/02/2026", "15/11/2026", "16/11/2026"],
    "city": ["Rio", "Niterói", None, None, "Rio", "São Gonçalo", "Rio", "Rio"],
}

df = pd.DataFrame(data)
df_original = df.copy()

In [64]:
# Diagnostics: display the shape, dtypes, number of nulls per column, and how
# many rows are duplicated.


print(df.shape)
print("\n----------------------------\n")
print(df.dtypes)
print("\n----------------------------\n")
print(df.isna().sum())
print("\n----------------------------\n")
print(f"There is/are {df.duplicated().sum()} duplicate row(s)")

(8, 5)

----------------------------

order_id     int64
customer    object
value       object
date        object
city        object
dtype: object

----------------------------

order_id    0
customer    1
value       0
date        0
city        2
dtype: int64

----------------------------

There is/are 1 duplicate row(s)


In [65]:
# Customer: standardize the customer column to have no leading/trailing spaces
# and the first letter of each name capitalized.

df["customer"] = df["customer"].str.strip().str.title()
print(df)

   order_id customer  value        date         city
0       201      Ana   49.9  10/11/2026          Rio
1       202    Bruno  249.9  11/11/2026      Niterói
2       203    Carla    n/a  12/11/2026         None
3       203    Carla    n/a  12/11/2026         None
4       204    Diego  119.9  13/11/2026          Rio
5       205     None   35.0  31/02/2026  São Gonçalo
6       206    Elisa    abc  15/11/2026          Rio
7       207      Ana   99.8  16/11/2026          Rio


In [66]:
# Value: convert the value column to numeric so anything that is not a number
# becomes NaN. Then, report how many NaNs it has.

df["value"] = pd.to_numeric(df["value"], errors="coerce")
print(df)
print("\n----------------------------\n")
print(f"There are {df['value'].isna().sum()} null values")


   order_id customer  value        date         city
0       201      Ana   49.9  10/11/2026          Rio
1       202    Bruno  249.9  11/11/2026      Niterói
2       203    Carla    NaN  12/11/2026         None
3       203    Carla    NaN  12/11/2026         None
4       204    Diego  119.9  13/11/2026          Rio
5       205     None   35.0  31/02/2026  São Gonçalo
6       206    Elisa    NaN  15/11/2026          Rio
7       207      Ana   99.8  16/11/2026          Rio

----------------------------

There are 3 null values


In [73]:
# Date: convert the date column to datetime (day/month/year format) so
# nonexistent dates become NaT. How many dates became NaT?

df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y", errors="coerce")
print(df)
print("\n----------------------------\n")
print(f"There are {df['date'].isna().sum()} null values")

   order_id customer  value       date           city
0       201      Ana   49.9 2026-11-10            Rio
1       202    Bruno  249.9 2026-11-11        Niterói
2       203    Carla   99.8 2026-11-12  Not specified
4       204    Diego  119.9 2026-11-13            Rio
5       205  Unknown   35.0        NaT    São Gonçalo
6       206    Elisa   99.8 2026-11-15            Rio
7       207      Ana   99.8 2026-11-16            Rio

----------------------------

There are 1 null values


In [68]:
# Duplicates: remove duplicate rows. How many rows are left?

df = df.drop_duplicates()
print(df)
print("\n----------------------------\n")
print(f"{len(df)} rows remaining.")


   order_id customer  value       date         city
0       201      Ana   49.9 2026-11-10          Rio
1       202    Bruno  249.9 2026-11-11      Niterói
2       203    Carla    NaN 2026-11-12         None
4       204    Diego  119.9 2026-11-13          Rio
5       205     None   35.0        NaT  São Gonçalo
6       206    Elisa    NaN 2026-11-15          Rio
7       207      Ana   99.8 2026-11-16          Rio

----------------------------

7 rows remaining.


In [69]:
# Nulls: in customer, fill with "unknown". In city, fill with "not provided".
# In value, fill with the median of valid values.

df.loc[:, "customer"] = df["customer"].fillna("Unknown")
df.loc[:, "city"] = df["city"].fillna("Not specified")
df.loc[:, "value"] = df["value"].fillna(df["value"].median())
print(df)

   order_id customer  value       date           city
0       201      Ana   49.9 2026-11-10            Rio
1       202    Bruno  249.9 2026-11-11        Niterói
2       203    Carla   99.8 2026-11-12  Not specified
4       204    Diego  119.9 2026-11-13            Rio
5       205  Unknown   35.0        NaT    São Gonçalo
6       206    Elisa   99.8 2026-11-15            Rio
7       207      Ana   99.8 2026-11-16            Rio


In [70]:
# Final check: display the dtypes and the number of nulls per column. Nulls
# should only remain where you decided not to fill them. Which column is that,
# and why wouldn't it make sense to fill it?

print(df.dtypes)
print("\n----------------------------\n")
print(df.isnull().sum())



order_id             int64
customer            object
value              float64
date        datetime64[ns]
city                object
dtype: object

----------------------------

order_id    0
customer    0
value       0
date        1
city        0
dtype: int64


In [72]:
# BONUS: After cleaning, calculate the mean of the value column and the number
# of distinct customers. Think about how the result would change if you had
# skipped step 2 (standardizing names).

media = df["value"].mean()
print(f"Média da coluna valores: {media}")
print("\n----------------------------\n")
print(f"Clientes únicos: {df["customer"].nunique()}")


Média da coluna valores: 107.72857142857141

----------------------------

Clientes únicos: 6
